# ReadBuddy captured-trace CNN adapter (pilot)

Upload `readbuddy_captured_adapter_colab_v2.zip`, then use **Runtime → Run all**. The notebook preserves the 455-output app contract and refuses to describe the one-writer captured data as test accuracy.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil

uploaded = files.upload()
package_name = next((name for name in uploaded if name.endswith('.zip')), None)
assert package_name, 'Upload readbuddy_captured_adapter_colab_v2.zip'
root = Path('/content/readbuddy_adapter')
if root.exists():
    shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(package_name) as archive:
    archive.extractall(root)
print('Extracted:', root)

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
assert tuple(int(x) for x in tf.__version__.split('.')[:2]) >= (2, 18), 'Use a current Colab runtime'

In [ ]:
import subprocess, sys
command = [
    sys.executable, str(root / 'adapt_captured_traces.py'),
    '--captured-dir', str(root / 'captured_dataset'),
    '--replay-dir', str(root / 'replay'),
    '--base-model', str(root / 'base' / 'open_set_cnn.keras'),
    '--output-dir', '/content/readbuddy_adapter_output',
    '--epochs', '18', '--capture-repeats', '60', '--seed', '42'
]
subprocess.run(command, check=True)

In [ ]:
import json
report_path = Path('/content/readbuddy_adapter_output/captured_adapter_report.json')
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps({
    'deploymentGatePassed': report['deploymentGatePassed'],
    'before': report['before'],
    'after': report['after'],
    'heldOutAccuracyDrop': report['heldOutAccuracyDrop'],
    'captureAudit': {k: report['captureAudit'][k] for k in ['totalManifestSamples','mappedSamples','writerCount']},
    'tfliteParity': report['tfliteParity'],
}, indent=2, ensure_ascii=False))
if not report['deploymentGatePassed']:
    print('STOP: gate failed. Do not replace the Flutter model.')
else:
    print('Gate passed. Download the evidence bundle for app-side validation.')

In [ ]:
archive = shutil.make_archive('/content/readbuddy_captured_adapter_output', 'zip', '/content/readbuddy_adapter_output')
files.download(archive)